# Installing Packages

In [73]:
import numpy as np
import torch
import matplotlib.pyplot as plt

%matplotlib qt
import scipy as sp

# necessary for linear algebra
torch.backends.cuda.preferred_linalg_library("magma")
# from torchdiffeq import odeint

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

Things to look at:
- https://jparkhill.netlify.app/solvingdiffusions/
- nangs for pytorch
- https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html
- https://developer.nvidia.com/gpugems/gpugems2/part-vi-simulation-and-numerical-algorithms/chapter-44-gpu-framework-solving

In general, we want to solve the equation
$$\left(\frac{\hbar^2}{2m}\left( -i\nabla + \mathbf{k} \right)^2 + V(\mathbf{r})\right)u_k(\mathbf{r}) = E(k)u_k(\mathbf{r})$$
with periodic BCs $u_k(\mathbf{r} + \mathbf{R})$.

For simplicity, let's start with 1D only, then the requation reduces to
$$\left(\frac{\hbar^2}{2m}\left( -i\frac{\partial}{\partial x} + k \right)^2 + V(x)\right)u_k(x) = E(k)u_k(x)$$
with $u_k(x) = u_k(x + a)$ and $\Psi(x) = e^{ikx}u_k(x)$

# 1D Solver

If we expand and apply the derivatives, and set $\hbar = m = 1$ in the diffeq above, the equation has the form
$$-u'' + k^2u - 2iku' + 2(V(x) - E)u = 0$$
Now we convert this into a system of 2 first-order ODEs by setting ($u(x) = y(x) = y_0)$
$$y_0 = u$$
$$y_1 = \dot{y}_0 = u'$$
$$\dot{y}_1 = u''$$
Thus, we obtain the following differential equations:
$$\dot{y}_0 = y_1$$
$$\dot{y}_1 = -2iky_1 + (2(V(t) - E) + k^2)y_0$$
just by making the replacement $x\rightarrow t$ since the odesolver treats the independent variable as time.

We can write this as a homogeneous vector-valued differential equation, taking
$$\frac{d}{dt}\mathbf{y} = A\mathbf{y}$$
with $A$ as a matrix.

Note that the usual matrix exponential is not a valid solution for periodic systems. Instead, we can use the Floquet theorem^ to solve for the **fundamental matrix** $Y$ that satisfies the equation (which can be easily solved by taking the matrix exponential)
$$\frac{d}{dt}Y(t) = A(t)Y(t)$$
with
$$Y(t) = Q(t)e^{Bt}$$
$$Y(0) = I$$

By using the Lyapunov-Floquet Transformation $y = Q(t)q$ (introducing transformed coordinate q), we have
$$B = \frac{1}{T}ln(Y(T))$$
$$Q(t) = Y(t)e^{-Bt}$$
$$\dot{q} = Bq$$

Finally, with $Y(t)$ and $q(t)$ computed by their known solutions, and $Q(t)$ and $B$ computed by the known formulas, we can write down the solution for y(t), parametrized by $k$ and $E$.

The last step is to find the actual values of $E$ that obey the quantization condition, which we can do by noting that the trace of $Y(t)$ is invariant (is this actually true?), thus $\mathrm{Tr}(Y(t)) = \mathrm{Tr}(Y(0)) = 2$, leading to the function we will minimize with respect to $E$:
$$|\mathrm{Tr}(Y(t)) - 2|$$

But we'll have better-behaved results if we instead use the signed optimizing function
$$\ln\left(\frac{1}{2}\left\vert\mathrm{Tr}(Y(t))\right\vert\right)$$


^see https://www.12000.org/my_notes/liapunov_floquet_transformation/bMATH_2018_FolkersE.pdf

So far we have e.v.'s from $\det \left(\lambda - \left\vert \prod_{n = 1}^{N} A_{n}(E) \right\vert ^{1/N} \right) = 0$, ie $\lambda_{\pm}(E)$. Then, take the grid and define for every site a local derivative (another factor of 1024), i.e.

$$\alpha(E) = \frac{\vert\lambda_{\pm}(E +\epsilon) - \lambda_{\pm}(E)\vert}{\epsilon}$$

and look for the sign flips!

In [74]:
# everything is taking the form [[y_0(t)],
#                                [y_1(t)]]

"""Global Parameters"""

# initial conditions on u(0) and u'(0)
initial_conditions = torch.tensor([[1], [0]])

x_max = 4
# the code is not built to take negative x values, Y(x_min = 0) = I is forced
x_min = 0
x_steps = 2048
x_vals = torch.linspace(0, x_max, x_steps + 1, device=device)

period = x_max

params = {
    "x_max": x_max,
    "x_min": x_min,
    "x_vals": x_vals,
    "x_steps": x_steps,
    "T": period,
    "y0": initial_conditions,
}

"""Defining Functions"""


def V(t):
    sinc_term = torch.nan_to_num(
        (
            torch.sin(x_max * torch.pi / (t - x_max/2))**2
            * ((t - x_max/2) / (x_max * torch.pi))**2
        ),
        nan=0
    )
    anharmonic_term = 4 * (1/2) * ((t - x_max/4)**2) * ((t - 3*x_max/4)**2)
    return (1 + 4.51568 * ((x_max * torch.pi)**2) * sinc_term)*anharmonic_term
    # return (1/2)*((t - x_max/4)**2)*((t - 3*x_max/4)**2)
    # return (1/2)*16*(t - x_max/2)**2


def A(t, k, E):
    a = torch.zeros(
        (2, 2) + t.shape + E.shape, dtype=torch.complex64, device=device
    )
    a[0, 0] = 0
    a[0, 1] = 1
    a[1, 0] = (
        2
        * (
            V(t).unsqueeze(-1).expand(t.shape + E.shape)
            - E.unsqueeze(0).expand(t.shape + E.shape)
        )
        + k**2
    )
    a[1, 1] = -2j * k
    return a.movedim((-2, -1), (0, 1))


def signChange(matrices):
    signed_matrices = torch.sign(matrices)
    # takes adjacent elements and multiplies, if <= 0 then a sign change occured
    change_index = torch.argwhere(
        signed_matrices[:-1] * signed_matrices[1:] <= 0
    ).squeeze(dim=1)
    return change_index


def trace_2by2(matrices):
    return matrices[:, 0, 0] + matrices[:, 1, 1]


class LyapunovFloquetSolver:
    # TODO: add some normalization/rescaling on A or its derivatives to make it more numerically stable
    # actually, only on the A's, use unitary assumption + trace condition to solve for eigenvalues?

    def __init__(self, k, params):
        self.k = k

        self.T = params["T"]

        self.x_max = params["x_max"]
        self.x_min = params["x_min"]
        self.x_steps = params["x_steps"]

        self.dx = (self.x_max + self.x_min) / (self.x_steps + 1)
        # throwing out first element correspoonding to Y(0) = I
        self.x_vals = params["x_vals"][1:]
        self.full_x_vals = params["x_vals"]
        self.y0 = params["y0"]

    def construct_Y(self, E):
        a = (A(self.x_vals, self.k, E)).movedim((0, 1), (-2, -1))
        a_exp = torch.zeros(a.shape, dtype=torch.complex64, device=device)
        delta = torch.sqrt(a[1, 0] - self.k**2)
        delta_expanded = delta.unsqueeze(0).unsqueeze(0).expand(a_exp.shape)

        a_exp[0, 0] = torch.cosh(
            delta * self.dx
        ) + 1j * self.k * torch.nan_to_num(
            torch.sinh(delta * self.dx) / delta, nan=self.dx
        )
        a_exp[1, 1] = torch.cosh(
            delta * self.dx
        ) - 1j * self.k * torch.nan_to_num(
            torch.sinh(delta * self.dx) / delta, nan=self.dx
        )
        a_exp[1, 0] = torch.sinh(delta * self.dx) * delta + torch.nan_to_num(
            torch.sinh(delta * self.dx) / delta, nan=self.dx
        ) * (self.k**2)
        a_exp[0, 1] = torch.nan_to_num(
            torch.sinh(delta * self.dx) / delta, nan=self.dx
        )

        # normalzing a_exp by multiplying by e^-delta
        a_exp /= torch.abs(torch.exp(delta_expanded * self.dx))
        return a_exp.movedim((0, 1), (-2, -1))

    def construct_Yinv(self, E):
        a = (A(self.x_vals, self.k, E)).movedim((0, 1), (-2, -1))
        a_exp = torch.zeros(a.shape, dtype=torch.complex64, device=device)
        delta = torch.sqrt(a[1, 0] - self.k**2)
        delta_expanded = delta.unsqueeze(0).unsqueeze(0).expand(a_exp.shape)

        a_exp[1, 1] = (
            torch.cosh(delta * self.dx)
            + 1j * self.k * torch.sinh(delta * self.dx) / delta
        )
        a_exp[0, 0] = (
            torch.cosh(delta * self.dx)
            - 1j * self.k * torch.sinh(delta * self.dx) / delta
        )
        a_exp[1, 0] = -1 * (
            torch.sinh(delta * self.dx) * (delta + ((self.k**2) / delta))
        )
        a_exp[0, 1] = -1 * torch.sinh(delta * self.dx) / delta

        # normalizing a_exp_inv by multiplying by e^delta (so it cancels with a_exp)
        a_exp *= torch.abs(torch.exp(delta_expanded * self.dx))
        return a_exp.movedim((0, 1), (-2, -1))

    def construct_derivative_Y(self, E):
        da = (A(self.x_vals, self.k, E)).movedim((0, 1), (-2, -1))
        da_exp = torch.zeros(da.shape, dtype=torch.complex64, device=device)
        da_exp_N = torch.zeros(da.shape, dtype=torch.complex64, device=device)
        da_exp_t = torch.zeros(da.shape, dtype=torch.complex64, device=device)
        delta = torch.sqrt(da[1, 0] - self.k**2)

        da_exp[0, 0] = (
            torch.sinh(delta * self.dx)
            + 1j * self.k * torch.cosh(delta * self.dx) / delta
        )
        da_exp[1, 1] = (
            torch.sinh(delta * self.dx)
            - 1j * self.k * torch.cosh(delta * self.dx) / delta
        )
        da_exp[1, 0] = torch.cosh(delta * self.dx) * (
            delta + ((self.k**2) / delta)
        )
        da_exp[0, 1] = torch.cosh(delta * self.dx) / delta

        da_exp *= -1 * self.dx / (delta)

        da_exp_N[0, 0] = 1j * self.k * torch.sinh(delta * self.dx) / delta
        da_exp_N[1, 1] = -1j * self.k * torch.sinh(delta * self.dx) / delta
        da_exp_N[1, 0] = torch.sinh(delta * self.dx) * (
            delta + ((self.k**2) / delta)
        )
        da_exp_N[0, 1] = torch.sinh(delta * self.dx) / delta

        da_exp_N *= 1 / (delta**2)

        da_exp_t[1, 0] = (-2 / delta) * torch.sinh(delta * self.dx)

        sum = da_exp + da_exp_N + da_exp_t

        a = (A(self.x_vals, self.k, E)).movedim((0, 1), (-2, -1))
        a_exp = torch.zeros(a.shape, dtype=torch.complex64, device=device)
        delta = torch.sqrt(a[1, 0] - self.k**2)
        delta_expanded = delta.unsqueeze(0).unsqueeze(0).expand(a_exp.shape)

        a_exp[0, 0] = (
            torch.cosh(delta * self.dx)
            + 1j * self.k * torch.sinh(delta * self.dx) / delta
        )
        a_exp[1, 1] = (
            torch.cosh(delta * self.dx)
            - 1j * self.k * torch.sinh(delta * self.dx) / delta
        )
        a_exp[1, 0] = torch.sinh(delta * self.dx) * (
            delta + ((self.k**2) / delta)
        )
        a_exp[0, 1] = torch.sinh(delta * self.dx) / delta

        a_exp /= torch.abs(torch.exp(delta_expanded * self.dx))

        normalized_sum = sum * torch.abs(
            torch.exp(-1 * delta * self.dx)
        )  # + torch.abs((self.dx/delta))*a_exp

        # normalized a_exp by muliplying e^-delta, propagated into derivative
        # return (torch.abs(torch.exp(-1*delta_expanded*self.dx))*sum).movedim((0, 1), (-2, -1))
        return normalized_sum.movedim((0, 1), (-2, -1))

    def construct_double_derivative_Y(self, E):
        a = (A(self.x_vals, self.k, E)).movedim((0, 1), (-2, -1))
        a_exp = torch.zeros(a.shape, dtype=torch.complex64, device=device)
        delta = torch.sqrt(a[1, 0] - self.k**2)

        a_exp[0, 0] = (
            torch.cosh(delta * self.dx)
            + 1j * self.k * torch.sinh(delta * self.dx) / delta
        )
        a_exp[1, 1] = (
            torch.cosh(delta * self.dx)
            - 1j * self.k * torch.sinh(delta * self.dx) / delta
        )
        a_exp[1, 0] = torch.sinh(delta * self.dx) * (
            delta + ((self.k**2) / delta)
        )
        a_exp[0, 1] = torch.sinh(delta * self.dx) / delta

        ######################

        da = (A(self.x_vals, self.k, E)).movedim((0, 1), (-2, -1))
        da_exp = torch.zeros(da.shape, dtype=torch.complex64, device=device)
        da_exp_N = torch.zeros(da.shape, dtype=torch.complex64, device=device)
        da_exp_t = torch.zeros(da.shape, dtype=torch.complex64, device=device)
        delta = torch.sqrt(da[1, 0] - self.k**2)
        delta_expanded = delta.unsqueeze(0).unsqueeze(0).expand(a_exp.shape)

        da_exp[0, 0] = (
            torch.sinh(delta * self.dx)
            + 1j * self.k * torch.cosh(delta * self.dx) / delta
        )
        da_exp[1, 1] = (
            torch.sinh(delta * self.dx)
            - 1j * self.k * torch.cosh(delta * self.dx) / delta
        )
        da_exp[1, 0] = torch.cosh(delta * self.dx) * (
            delta + ((self.k**2) / delta)
        )
        da_exp[0, 1] = torch.cosh(delta * self.dx) / delta

        da_exp *= -1 * self.dx / (delta)

        da_exp_N[0, 0] = 1j * self.k * torch.sinh(delta * self.dx) / delta
        da_exp_N[1, 1] = -1j * self.k * torch.sinh(delta * self.dx) / delta
        da_exp_N[1, 0] = torch.sinh(delta * self.dx) * (
            delta + ((self.k**2) / delta)
        )
        da_exp_N[0, 1] = torch.sinh(delta * self.dx) / delta

        da_exp_N *= 1 / (delta**2)

        da_exp_t[1, 0] = (-2 / delta) * torch.sinh(delta * self.dx)

        bprime = da_exp + da_exp_N + da_exp_t

        sum = (1 / (delta**2)) * (
            bprime
            + (self.dx**2) * a_exp
            + 2
            * (
                1
                - (
                    (delta * self.dx)
                    * torch.cosh(delta * self.dx)
                    / torch.sinh(delta * self.dx)
                )
            )
            * da_exp_N
            - da_exp_t
        )

        # normalized a_exp by muliplying by e^-delta, propagated into 2nd derivative
        # normalized_sum = (
        #     sum*torch.exp(-1*delta_expanded*self.dx)
        #     + (2*self.dx/delta_expanded)*torch.exp(-1*delta_expanded*self.dx)*bprime
        #     + ((self.dx**2)/(delta_expanded**2))*(1 + (1/delta_expanded))*torch.exp(-1*delta_expanded*self.dx)*a_exp
        # )
        normalized_sum = sum / torch.abs(torch.exp(delta_expanded * self.dx))

        return normalized_sum.movedim((0, 1), (-2, -1))

    def collapse_function(self, matrices):
        while matrices.shape[0] > 1:
            # computer number of steps_per_t currently in unitaries
            n_slices = matrices.shape[0]

            # separate unitaries into pairs, view has dims [x_steps // 2, 2, E_steps, 2, 2]
            matrices = matrices.view((n_slices // 2, 2) + matrices.shape[1:])

            # multiply pairs until number of unitaries reduced below time_steps
            matrices = torch.matmul(matrices[:, 0], matrices[:, 1])
        # removed e^-ik for testing
        # return np.exp(-1j*self.k)*matrices.squeeze(0)
        return matrices.squeeze(0)

    def collapse_Y(self, E):
        matrices = self.construct_Y(E)
        return self.collapse_function(matrices)

    def collapse_dY(self, E):
        matrices1 = self.construct_Y(E)
        matrices2 = self.construct_derivative_Y(E)

        # expand into x_steps x x_steps blocks
        matrices1_expanded = (
            matrices1.unsqueeze(1)
            .expand((self.x_steps,) + matrices1.shape)
            .clone()
        )
        # set diagonal of blocks with derivative
        matrices1_expanded[range(self.x_steps), range(self.x_steps)] = (
            matrices2
        )
        collapsed_matrices = self.collapse_function(matrices1_expanded)
        return torch.sum(collapsed_matrices, dim=0)

    def collapse_ddY(self, E):
        batch_size = 2**3  ## power of 2
        matrices1 = self.construct_Y(E)
        matrices2 = self.construct_derivative_Y(E)
        matrices3 = self.construct_double_derivative_Y(E)
        # output = torch.zeros(
        #     E.shape + (2,2), dtype=torch.complex64, device=device,
        # )

        # expand into x_steps x x_steps blocks
        matrices_expanded = (
            matrices1.unsqueeze(1)
            .expand((self.x_steps,) + matrices1.shape)
            .clone()
        )
        matrices_expanded[range(self.x_steps), range(self.x_steps)] = matrices2
        # set diagonal of blocks with derivative

        # matrices_expanded_batch = matrices_expanded.unsqueeze(2).expand(
        #     (self.x_steps//batch_size,) + matrices_expanded.shape
        # ).clone()

        total = torch.zeros(
            E.shape + (2, 2),
            dtype=torch.complex64,
            device=device,
        )

        for k in range(self.x_steps):
            matrices_expanded[k, range(self.x_steps)] = matrices2
            matrices_expanded[k, k] = matrices3[k]
            collapsed_matrices = self.collapse_function(matrices_expanded)
            total += torch.sum(collapsed_matrices, dim=0)
        return total

    def loss(self, E):
        if type(E) is not torch.Tensor:
            E = torch.tensor([E], device=device)

        a = (A(self.x_vals, self.k, E)).movedim((0, 1), (-2, -1))
        delta = torch.sqrt(a[1, 0] - self.k**2)
        norm = (torch.sum(torch.real(delta * self.dx), dim=0)).squeeze()
        target = 2 * np.cos(self.k) * torch.exp(-1 * norm)

        matrices = self.collapse_Y(E)

        loss = torch.real(trace_2by2(matrices)) * torch.exp(norm)  # - target
        # print(f"Target = {target}")

        return torch.nan_to_num(loss)

    def im_loss(self, E):
        if type(E) is not torch.Tensor:
            E = torch.tensor([E], device=device)

        matrices = self.collapse_Y(E)

        loss = torch.imag(trace_2by2(matrices))

        return torch.nan_to_num(loss)

    def derivative_loss(self, E):
        if type(E) is not torch.Tensor:
            E = torch.tensor([E], device=device)

        a = (A(self.x_vals, self.k, E)).movedim((0, 1), (-2, -1))
        delta = torch.sqrt(a[1, 0] - self.k**2)
        norm = (torch.sum(torch.real(delta * self.dx), dim=0)).squeeze()

        d_matrices = self.collapse_dY(E)

        loss = torch.real(trace_2by2(d_matrices)) * torch.exp(norm)

        return torch.nan_to_num(loss)

    def dderivative_loss(self, E):
        if type(E) is not torch.Tensor:
            E = torch.tensor([E], device=device)

        dd_matrices = self.collapse_ddY(E)

        a = (A(self.x_vals, self.k, E)).movedim((0, 1), (-2, -1))
        delta = torch.sqrt(a[1, 0] - self.k**2)
        norm = (torch.sum(torch.real(delta * self.dx), dim=0)).squeeze()

        loss = torch.real(trace_2by2(dd_matrices)) * torch.exp(norm)
        return torch.nan_to_num(loss)

    def construct_eigenstate_sym(self, E, tol=10):
        if type(E) is not torch.Tensor:
            E = torch.tensor([E], device=device)

        eigenstate = torch.zeros(
            (self.x_steps + 1,) + E.shape + (2, 1),
            dtype=torch.complex64,
            device=device,
        )

        matrices = self.construct_Y(E)
        # NOTE: included identity as the first matrix
        matrices_inv = torch.concat(
            (
                torch.eye(2, device=device)
                .unsqueeze(0)
                .expand(E.shape + (2, 2))
                .unsqueeze(0),
                self.construct_Yinv(E),
            ),
            dim=0,
        )
        midpoint = (self.x_steps + 1) // 2
        # eigenstate[midpoint] = self.y0.unsqueeze(0).expand(E.shape + (2,1))
        evals, evects = torch.linalg.eig(matrices[midpoint - 1])

        eigenstate[midpoint] = evects[..., 0].unsqueeze(-1)

        diff_max = 0

        for i in range(self.x_steps // 2):
            # right eigenstates
            eigenstate[midpoint + i + 1] = torch.matmul(
                matrices[midpoint + i - 1], eigenstate[midpoint + i]
            )
            norm_next_right = torch.sqrt(
                torch.sum(torch.abs(eigenstate[midpoint + i + 1]) ** 2, dim=-2)
            )
            norm_now_right = torch.sqrt(
                torch.sum(torch.abs(eigenstate[midpoint + i]) ** 2, dim=-2)
            )

            # left eigenstates
            eigenstate[midpoint - i - 1] = torch.matmul(
                matrices_inv[midpoint - i], eigenstate[midpoint - i]
            )
            norm_next_left = torch.sqrt(
                torch.sum(torch.abs(eigenstate[midpoint - i - 1]) ** 2, dim=-2)
            )
            norm_now_left = torch.sqrt(
                torch.sum(torch.abs(eigenstate[midpoint - i]) ** 2, dim=-2)
            )

            # if trace_right < 2 and trace_left < 2:
            eigenstate[midpoint + i + 1] *= norm_now_right / norm_next_right
            eigenstate[midpoint - i - 1] *= norm_now_left / norm_next_left

            # elif trace_right > 2 and trace_left < 2:
            #     eigenstate[midpoint - i - 1] *= norm_now_left / norm_next_left

            # elif trace_right < 2 and trace_left > 2:
            #     eigenstate[midpoint + i + 1] *= norm_now_right / norm_next_right

            diff = torch.abs(
                torch.abs(eigenstate[midpoint + i + 1, ..., 0, 0])
                - torch.abs(eigenstate[midpoint - i - 1, ..., 0, 0])
            ).squeeze()
            assert diff.item() < tol, (
                f"Eigenstate not symmetric with asymmetry {diff.item()} > {tol}"
            )
            if diff > diff_max:
                diff_max = diff

        # print(diff_max)
        return (
            torch.exp(1j * self.k * self.full_x_vals)
            .unsqueeze(1)
            .expand((self.x_steps + 1,) + E.shape)
            * eigenstate[..., 0, 0]
        )

    def construct_eigenstate_asym(self, E):
        if type(E) is not torch.Tensor:
            E = torch.tensor([E], device=device)

        eigenstate = torch.zeros(
            (self.x_steps + 1,) + E.shape + (2, 1),
            dtype=torch.complex64,
            device=device,
        )

        matrices = self.construct_Y(E)
        evals, evects = torch.linalg.eig(matrices[0])

        eigenstate[0] = evects[..., 0].unsqueeze(-1)
        for i in range(self.x_steps):
            eigenstate[i + 1] = torch.matmul(matrices[i], eigenstate[i])

            norm_next = torch.sqrt(
                torch.sum(torch.abs(eigenstate[i + 1]) ** 2, dim=-2)
            )
            norm_now = torch.sqrt(
                torch.sum(torch.abs(eigenstate[i]) ** 2, dim=-2)
            )

            eigenstate[i + 1] *= norm_now / norm_next

        return (
            torch.exp(1j * self.k * self.full_x_vals)
            .unsqueeze(1)
            .expand((self.x_steps + 1,) + E.shape)
            * eigenstate[..., 0, 0]
        )

In [75]:
k = np.linspace(0, 2 * np.pi, 2048)
initial_guess = 2.1
energies = torch.linspace(1.998, 2.002, 1024, device=device)
solver_zero = LyapunovFloquetSolver(0, params)

In [76]:
# eigenstate_grid = torch.empty(k.shape + (x_steps + 1,), device=device)
# eigenvals_loss = np.zeros(k.shape)
# loss_guess = initial_guess

# for i in range(len(k)):
#     solver_loss = LyapunovFloquetSolver(k[i], params)

#     try:
#         eigenval = sp.optimize.newton(
#             lambda E: solver_zero.loss(E).item() - 2 * np.cos(k[i]), loss_guess
#         )
#         eigenvals_loss[i] = eigenval
#         loss_guess = eigenval
#     except RuntimeError:
#         eigenvals_loss[i] = eigenvals_loss[i - 1]

# # eigenvals_dispersion = np.zeros_like(eigenvals_loss)
# # eigenvals_dispersion[0] = sp.optimize.newton(lambda E: solver_zero.loss(E).item(), initial_guess)

# # for i in range(1, len(k)):
# #     solver_disp = LyapunovFloquetSolver(k[i - 1], params)
# #     dispersion = (2*(x_min - x_max)*np.sin(k[i - 1]))/solver_disp.derivative_loss(eigenvals_dispersion[i - 1])
# #     eigenvals_dispersion[i] = eigenvals_dispersion[i - 1] +  dispersion*((2*np.pi)/len(k))

In [77]:
# fig, ax = plt.subplots(figsize=(10, 8))

# ax.plot(k, eigenvals_loss, label="Loss Eigenvalues", color="tab:orange")
# # ax.plot(k, eigenvals_dispersion, label="Dispersion Eigenvalues", color="tab:blue")

# ax.set_title(f"Band Structure Plot for Eigenvalue Guess = {initial_guess}")
# ax.grid()
# ax.legend()

# ax.set_xlabel("k")
# ax.set_ylabel("E")

# plt.show()

In [78]:
single_solver = LyapunovFloquetSolver(0, params)
loss_single = single_solver.loss(energies).cpu()
# loss_single_im = single_solver.im_loss(energies).cpu()
# loss_derivative_single = single_solver.derivative_loss(energies).cpu()
# loss_dderivative_single = single_solver.dderivative_loss(energies).cpu()
# # notice for x_max = 1, x_steps = 1024 it is ok
# # but if we increse x_max=8, x_steps=8*1024 it has error

In [79]:
cpu_energies = energies.cpu()

fig, ax = plt.subplots(figsize=(10, 8))

# ax2 = ax.twinx()

ax.plot(cpu_energies, loss_single, label="Losses", color="tab:orange")
# ax.plot(cpu_energies, loss_single_im, label=f" Imaginary Losses k = {k[-1]:.2f}", color="tab:blue")
# ax.plot(
#     cpu_energies,
#     loss_derivative_single,
#     label="Derivative Losses",
#     color="tab:green"
# )
# ax.plot(
#     cpu_energies,
#     loss_dderivative_single,
#     label="2nd Derivative Losses",
#     color="tab:blue"
# )

ax.legend()
ax.grid()
ax.set_title("Losses vs. Energy")

ax.set_xlabel("Energy")
# ax2.set_ylabel("Loss")s
ax.set_ylabel("Loss")

# ax.set_xlim(cpu_energies.min(), cpu_energies.max())
# ax.set_xlim(0.5002, 0.5003)
# ax.set_ylim(-0.0001, 0.0001)

# ax.set_yscale("log")

plt.show()

In [80]:
# fig, ax = plt.subplots(figsize=(10,8))

# x_cpu = x_vals.cpu()

# # energy = sp.optimize.newton(lambda E: .loss(E).item(), 0.4)
# # eigenstate = test.construct_eigenstate_sym(energy).cpu().squeeze()

# for i in range(len(k)):
#     if i % 100 == 50:
#         ax.plot(x_cpu, torch.abs(eigenstate_grid[i].cpu())**2, label=f"E = {eigenvals[i]}, k = {k[i]}")
# # ax.plot(x_vals.cpu(), torch.abs(eigenstate)**2, label=f"E = {energy}")

# ax.legend()
# ax.grid()
# ax.set_title("Eigenstate Plot")

# ax.set_xlabel("x")

# # ax.set_yscale("log")

# plt.show()